# Road Accidents France — Data Cleaning Pipeline

This notebook cleans and merges the 4 raw BAAC tables from [data.gouv.fr](https://www.data.gouv.fr) for a given year.

**Source tables:**
- `Caract_{YEAR}.csv` — accident characteristics (date, location, weather conditions)
- `Lieux_{YEAR}.csv` — road and location details
- `Vehicules_{YEAR}.csv` — vehicle information
- `Usagers_{YEAR}.csv` — user/victim information

**Output:** Two clean CSVs ready for EDA and ML:
- `accidents_france_{YEAR}_clean_for_EDA.csv` — with human-readable labels
- `accidents_france_{YEAR}_clean_for_ML.csv` — with original numeric codes

## 0. Configuration

Change `YEAR` to process a different year.

In [ ]:
import pandas as pd
import numpy as np

YEAR = 2023  # Change to 2024 to process 2024 data

## 1. Load Raw Data

In [ ]:
df_caract = pd.read_csv(f'Caract_{YEAR}.csv', sep=';', encoding='latin-1')
df_lieux = pd.read_csv(f'Lieux_{YEAR}.csv', sep=';', encoding='latin-1')
df_vehicules = pd.read_csv(f'Vehicules_{YEAR}.csv', sep=';', encoding='latin-1')
df_usagers = pd.read_csv(f'Usagers_{YEAR}.csv', sep=';', encoding='latin-1')

print(f"Caract: {df_caract.shape}")
print(f"Lieux: {df_lieux.shape}")
print(f"Vehicules: {df_vehicules.shape}")
print(f"Usagers: {df_usagers.shape}")

## 2. Merge Tables

All tables share `Num_Acc` as the accident identifier.
Vehicules and Usagers are also linked via `id_vehicule`.

We merge Vehicules + Usagers first to avoid a cartesian product.

In [ ]:
# Deduplicate Lieux (some accidents have multiple location entries)
df_lieux = df_lieux.drop_duplicates(subset='Num_Acc', keep='first')

# Merge Vehicules + Usagers first via Num_Acc and id_vehicule
df_veh_usa = df_vehicules.merge(df_usagers, on=['Num_Acc', 'id_vehicule'], how='left')

# Merge Caract + Lieux
df = df_caract.merge(df_lieux, on='Num_Acc', how='left')

# Merge with Vehicules + Usagers
df = df.merge(df_veh_usa, on='Num_Acc', how='left')

print(f"Merged dataframe: {df.shape}")

## 3. Drop Irrelevant Columns

In [ ]:
cols_to_drop = ['adr', 'v1', 'v2', 'voie', 'pr', 'pr1', 'lartpc', 'larrout',
                'occutc', 'locp', 'etatp', 'secu2', 'secu3', 'place']

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"After dropping irrelevant columns: {df.shape}")

## 4. Handle Missing Values

In this dataset, `-1` encodes missing/unknown values. We replace them with `NaN`.

In [ ]:
# Check proportion of -1 per column before replacing
for col in df.columns:
    pct = (df[col] == -1).sum() / len(df) * 100
    if pct > 0:
        print(f"{col}: {pct:.1f}% missing (-1)")

df.replace(-1, np.nan, inplace=True)

## 5. Fix Data Types

In [ ]:
# Build datetime from separate columns
df['datetime'] = pd.to_datetime(
    df['an'].astype(str) + '/' +
    df['mois'].astype(str) + '/' +
    df['jour'].astype(str) + ' ' +
    df['hrmn'].astype(str),
    format='%Y/%m/%d %H:%M'
)
df.drop(columns=['an', 'mois', 'jour', 'hrmn'], inplace=True)

# Compute age from birth year
df['Age'] = YEAR - df['an_nais']
df.drop(columns=['an_nais'], inplace=True)

print(df.dtypes)

## 6. Drop Rows with Missing Values

We drop columns with too many missing values first, then drop remaining rows with NaN.

In [ ]:
# Drop columns with more than 30% missing values
threshold = 0.3
missing_pct = df.isnull().mean()
cols_high_missing = missing_pct[missing_pct > threshold].index.tolist()
print(f"Dropping columns with >{threshold*100}% missing: {cols_high_missing}")
df.drop(columns=cols_high_missing, inplace=True)

before = len(df)
df.dropna(inplace=True)
after = len(df)
print(f"Dropped {before - after} rows ({(before-after)/before*100:.1f}%) with remaining NaN")
print(f"Final shape: {df.shape}")

## 7. Save ML Version

Before encoding labels, we save the numeric version for ML.

In [ ]:
df.to_csv(f'accidents_france_{YEAR}_clean_for_ML.csv', index=False)
print(f"Saved ML dataset: accidents_france_{YEAR}_clean_for_ML.csv")

## 8. Encode Labels for EDA

We map numeric codes to human-readable labels as per the BAAC documentation.

In [ ]:
# Injury severity
df['grav'] = df['grav'].map({1: 'Uninjured', 2: 'Killed', 3: 'Hospitalized', 4: 'Minor injury'})
df.rename(columns={'grav': 'Injury_Severity'}, inplace=True)

# Lighting conditions
df['lum'] = df['lum'].map({
    1: 'Daylight', 2: 'Dusk/Dawn', 3: 'Night no lighting',
    4: 'Night lighting off', 5: 'Night lighting on'
})
df.rename(columns={'lum': 'Lighting'}, inplace=True)

# Weather conditions
df['atm'] = df['atm'].map({
    1: 'Normal', 2: 'Light rain', 3: 'Heavy rain', 4: 'Snow/Hail',
    5: 'Fog/Smoke', 6: 'Strong wind', 7: 'Dazzling', 8: 'Overcast', 9: 'Other'
})
df.rename(columns={'atm': 'Weather'}, inplace=True)

# Collision type
df['col'] = df['col'].map({
    1: 'Head-on (2 vehicles)', 2: 'Rear-end (2 vehicles)', 3: 'Side (2 vehicles)',
    4: 'Chain (3+ vehicles)', 5: 'Multiple (3+ vehicles)', 6: 'Other', 7: 'No collision'
})
df.rename(columns={'col': 'Collision_Type'}, inplace=True)

# Road category
df['catr'] = df['catr'].map({
    1: 'Motorway', 2: 'National road', 3: 'Departmental road',
    4: 'Municipal road', 5: 'Off public network',
    6: 'Public parking', 7: 'Urban road', 9: 'Other'
})
df.rename(columns={'catr': 'Road_Category'}, inplace=True)

# User category
df['catu'] = df['catu'].map({1: 'Driver', 2: 'Passenger', 3: 'Pedestrian'})
df.rename(columns={'catu': 'User_Category'}, inplace=True)

# Sex
df['sexe'] = df['sexe'].map({1: 'Male', 2: 'Female'})
df.rename(columns={'sexe': 'Sex'}, inplace=True)

# Vehicle category (simplified)
df['catv'] = df['catv'].map({
    0: 'Undetermined', 1: 'Bicycle', 2: 'Moped <50cc', 3: 'Microcar',
    7: 'Car', 10: 'Light van', 13: 'HGV 3.5-7.5T', 14: 'HGV >7.5T',
    15: 'HGV + trailer', 16: 'Truck solo', 17: 'Truck + semi',
    20: 'Special vehicle', 21: 'Farm tractor',
    30: 'Scooter <50cc', 31: 'Motorcycle 50-125cc', 32: 'Scooter 50-125cc',
    33: 'Motorcycle >125cc', 34: 'Scooter >125cc',
    37: 'Bus', 38: 'Coach', 39: 'Train', 40: 'Tram',
    50: 'Motorized scooter', 60: 'Non-motorized scooter', 80: 'E-bike', 99: 'Other'
})
df.rename(columns={'catv': 'Vehicle_Category'}, inplace=True)

# Safety equipment
df['secu1'] = df['secu1'].map({
    0: 'None', 1: 'Seatbelt', 2: 'Helmet', 3: 'Child device',
    4: 'Reflective vest', 5: 'Airbag', 6: 'Gloves', 7: 'Gloves+Airbag',
    8: 'Undetermined', 9: 'Other'
})
df.rename(columns={'secu1': 'Safety_Equipment'}, inplace=True)

# Trip purpose
df['trajet'] = df['trajet'].map({
    0: 'Unknown', 1: 'Home-Work', 2: 'Home-School',
    3: 'Shopping', 4: 'Professional', 5: 'Leisure', 9: 'Other'
})
df.rename(columns={'trajet': 'Trip_Purpose'}, inplace=True)

# Rename lat/long
df.rename(columns={'lat': 'Latitude', 'long': 'Longitude'}, inplace=True)

print("Label encoding done.")
print(df.head())

## 9. Save EDA Version

In [ ]:
df.to_csv(f'accidents_france_{YEAR}_clean_for_EDA.csv', index=False)
print(f"Saved EDA dataset: accidents_france_{YEAR}_clean_for_EDA.csv")